In [18]:
import numpy as np
import plotly.graph_objects as go

In [19]:
# ============================================================
# 🎲 VARIABLE SEED: Genera resultados diferentes pero rastreables
# ============================================================
import numpy as np
import time

# Genera una semilla basada en el tiempo actual (microsegundos)
# Esto asegura que cada ejecución tenga resultados diferentes
RANDOM_SEED = int((time.time() * 1000000) % 100000)  # Semilla entre 0-99999
np.random.seed(RANDOM_SEED)

# Mensaje prominente para rastrear el rendimiento
print("🎲" + "="*60)
print(f"🎯 SEMILLA ACTUAL: {RANDOM_SEED}")
print("   ⚡ ANOTA ESTA SEMILLA SI OBTIENES BUENOS RESULTADOS")
print("   🔄 Para reproducir: cambia línea 8 a RANDOM_SEED = {0}".format(RANDOM_SEED))
print("="*62)

🎲============================================================
🎯 SEMILLA ACTUAL: 59034
   ⚡ ANOTA ESTA SEMILLA SI OBTIENES BUENOS RESULTADOS
   🔄 Para reproducir: cambia línea 8 a RANDOM_SEED = 59034


In [20]:
# ============================================================
# 1) True nonlinear system (Differential Drive Mobile Robot)
# ============================================================
def plant_dynamics(x, u, L=0.5, friction_coeff=0.1, slip_factor=0.05):
    """
    Continuous dynamics for differential drive mobile robot: x = [x_pos, y_pos, theta]. 
    Returns x_dot.
    
    The differential drive robot equations:
    dx/dt = v * cos(θ) 
    dy/dt = v * sin(θ)
    dθ/dt = ω
    
    where:
    v = (v_r + v_l) / 2  (linear velocity)
    ω = (v_r - v_l) / L  (angular velocity)
    L = wheelbase distanceconms
    """
    x_pos, y_pos, theta = x
    v_l, v_r = u  # left and right wheel velocities
    
    # Add realistic wheel slip effects
    v_l_actual = v_l * (1 - slip_factor * np.random.randn())
    v_r_actual = v_r * (1 - slip_factor * np.random.randn())
    
    # Compute linear and angular velocities
    v = (v_r_actual + v_l_actual) / 2.0
    omega = (v_r_actual - v_l_actual) / L
    
    # Add friction effects (velocity-dependent)
    v_friction = v * (1 - friction_coeff * np.abs(v))
    omega_friction = omega * (1 - friction_coeff * np.abs(omega))
    
    # Robot kinematics
    x_dot = v_friction * np.cos(theta)
    y_dot = v_friction * np.sin(theta)
    theta_dot = omega_friction
    
    return np.array([x_dot, y_dot, theta_dot])

def plant(x_k, u_k, dt=0.01, process_noise_type='mixed', process_noise_std=0.05, 
          terrain_roughness=0.02, sensor_bias=[0.0, 0.0, 0.0]):
    """
    One Euler step of the discrete plant with realistic mobile robot disturbances.
    
    Args:
        x_k: current state [x, y, theta]
        u_k: control input [v_left, v_right] 
        dt: time step
        process_noise_type: type of noise ('mixed', 'gaussian', 'laplacian')
        process_noise_std: standard deviation of process noise
        terrain_roughness: terrain-induced disturbances
        sensor_bias: systematic biases in measurements
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot
    
    # Realistic mobile robot disturbances
    
    # 1. Terrain-induced disturbances (position-dependent)
    terrain_noise = terrain_roughness * np.array([
        np.sin(0.5 * x_kp1[0]) * np.random.randn(),  # x-direction terrain variation
        np.cos(0.3 * x_kp1[1]) * np.random.randn(),  # y-direction terrain variation  
        0.1 * np.sin(x_kp1[2]) * np.random.randn()   # angular disturbance from terrain
    ])
    
    # 2. Velocity-dependent noise (increases with speed)
    velocity_magnitude = np.linalg.norm(u_k)
    velocity_noise_factor = 1 + 0.2 * velocity_magnitude
    
    # 3. Mixed process noise (combination of different noise types)
    if process_noise_type == 'mixed':
        # Gaussian component (main noise)
        gaussian_noise = np.random.normal(0, process_noise_std * velocity_noise_factor, size=x_kp1.shape)
        # Impulse noise (occasional large disturbances)
        impulse_prob = 0.02  # 2% chance of impulse noise
        impulse_noise = np.zeros_like(x_kp1)
        if np.random.rand() < impulse_prob:
            impulse_noise = np.random.normal(0, process_noise_std * 5, size=x_kp1.shape)
        # Laplacian component (heavy-tailed noise)
        laplacian_noise = np.random.laplace(0, process_noise_std * 0.3, size=x_kp1.shape)
        
        total_noise = gaussian_noise + impulse_noise + laplacian_noise
    elif process_noise_type == 'laplacian':
        total_noise = np.random.laplace(0, process_noise_std * velocity_noise_factor, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std * velocity_noise_factor
        total_noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        total_noise = np.random.normal(0, process_noise_std * velocity_noise_factor, size=x_kp1.shape)
    
    # 4. Systematic biases (drift, calibration errors)
    bias_noise = np.array(sensor_bias) * dt
    
    # 5. Wheel encoder quantization effects
    # Wheel encoder quantization (linear for x,y; angular for theta)
    enc_res_xy = 0.001                 # 1 mm
    enc_res_th = np.deg2rad(0.1)       # 0.1°
    quantization_noise = np.array([
        enc_res_xy * (np.random.rand() - 0.5),
        enc_res_xy * (np.random.rand() - 0.5),
        enc_res_th * (np.random.rand() - 0.5),
    ])  

    
    # Combine all disturbances
    x_kp1 += terrain_noise + total_noise + bias_noise + quantization_noise
    
    # 6. Angle wrapping for realistic behavior
    x_kp1[2] = np.arctan2(np.sin(x_kp1[2]), np.cos(x_kp1[2]))  # wrap angle to [-π, π]
    
    return x_kp1

def generate_realistic_trajectory(t, trajectory_type='straight'):
    """
    Generate realistic control inputs for mobile robot.
    
    Args:
        t: time value
        trajectory_type: 'straight', 'circle', 'figure8', 'mixed', 'obstacle_avoidance'
    
    Returns:
        u: [v_left, v_right] wheel velocities
    """
    if trajectory_type == 'straight':
        # Straight line with small variations
        v_base = 1.0 + 0.2 * np.sin(0.5 * t)
        return np.array([v_base, v_base])
    
    elif trajectory_type == 'circle':
        # Circular motion
        v_l = 1.0 + 0.1 * np.sin(t)
        v_r = 1.5 + 0.1 * np.cos(t)
        return np.array([v_l, v_r])
    
    elif trajectory_type == 'figure8':
        # Figure-8 pattern
        v_l = 1.0 + 0.8 * np.sin(0.5 * t)
        v_r = 1.0 - 0.8 * np.sin(0.5 * t)
        return np.array([v_l, v_r])
    
    elif trajectory_type == 'obstacle_avoidance':
        # Obstacle avoidance maneuvers
        base_speed = 1.2
        avoidance_maneuver = 0.5 * np.sin(2 * t) * np.exp(-0.1 * t)
        v_l = base_speed + avoidance_maneuver
        v_r = base_speed - avoidance_maneuver
        return np.array([v_l, v_r])
    
    else:  # 'mixed' - combination of different behaviors
        # Mixed trajectory with different phases
        phase = (t % 20.0) / 20.0  # 20-second cycles
        
        if phase < 0.3:  # Straight motion
            v_base = 1.5
            return np.array([v_base, v_base])
        elif phase < 0.6:  # Turning
            v_l = 0.8
            v_r = 1.8
            return np.array([v_l, v_r])
        elif phase < 0.8:  # Reverse
            v_base = -0.5
            return np.array([v_base, v_base])
        else:  # Complex maneuver
            v_l = 1.0 + 0.5 * np.sin(10 * t)
            v_r = 1.0 + 0.5 * np.cos(10 * t)
            return np.array([v_l, v_r])
        

# === NEW: trayectoria de referencia y error de postura ===
def reference_trajectory(t, kind='leminiscate'):
    """Devuelve (x_d, y_d, theta_d, v_ff, omega_ff) como feedforward opcional."""
    if kind == 'circle':
        R, w = 3.0, 0.2
        x_d = R * np.cos(w*t)
        y_d = R * np.sin(w*t)
        theta_d = np.arctan2(w*R*np.cos(w*t), -w*R*np.sin(w*t))  # tangencial
        v_ff = R * w
        omega_ff = w
        return np.array([x_d, y_d, theta_d]), v_ff, omega_ff
    elif kind == 'lemniscate':  # ∞
        a, w = 2.0, 0.15
        x_d = a * np.sin(w*t)
        y_d = a * np.sin(w*t) * np.cos(w*t)
        theta_d = np.arctan2(
            a*w*(np.cos(w*t)**2 - np.sin(w*t)**2),
            a*w*np.cos(w*t)
        )
        v_ff = a*w*np.sqrt(np.cos(w*t)**2 + (np.cos(w*t)**2 - np.sin(w*t)**2)**2)
        omega_ff = 0.0
        return np.array([x_d, y_d, theta_d]), v_ff, omega_ff
    elif kind == 'city_streets':
        # Complex city street pattern with multiple rectangular blocks
        # Similar to the pattern shown in the user's image
        
        # Define waypoints for city streets trajectory (in meters)
        # This creates a realistic urban driving pattern with smooth curved corners (no inner loops)
        corner_radius = 0.4  # Radius for corner curves (meters)
        waypoints = np.array([
            # Start and main circuit with curved corners
            [0.0, 0.0],      # Start
            [3.6, 0.0],      # Approach corner
            [3.8, 0.1],      # Corner curve start
            [3.9, 0.3],      # Corner curve
            [4.0, 0.6],      # Corner curve
            [4.0, 2.6],      # Straight section
            [4.0, 2.8],      # Corner curve start
            [4.2, 2.9],      # Corner curve
            [4.4, 3.0],      # Corner curve
            [5.6, 3.0],      # Approach corner
            [5.8, 3.1],      # Corner curve start
            [5.9, 3.3],      # Corner curve
            [6.0, 3.6],      # Corner curve end
            [6.0, 4.6],      # Straight section
            [6.0, 4.8],      # Corner curve start
            [5.9, 4.9],      # Corner curve
            [5.7, 5.0],      # Corner curve
            [4.4, 5.0],      # Approach corner
            [4.2, 4.9],      # Corner curve start
            [4.1, 4.7],      # Corner curve  
            [4.0, 4.4],      # Corner curve end (back to main)
            [4.0, 5.0],      # Continue to next corner
            [1.4, 5.0],      # Approach corner
            [1.2, 5.1],      # Corner curve start
            [1.1, 5.3],      # Corner curve
            [1.0, 5.6],      # Corner curve end
            [1.0, 6.6],      # Straight section
            [1.0, 6.8],      # Corner curve start
            [0.9, 6.9],      # Corner curve
            [0.7, 7.0],      # Corner curve
            [-0.6, 7.0],     # Approach corner
            [-0.8, 6.9],     # Corner curve start
            [-0.9, 6.7],     # Corner curve
            [-1.0, 6.4],     # Corner curve end
            [-1.0, 5.4],     # Straight section (back to main)
            [-1.6, 5.0],     # Continue west
            [-1.8, 4.9],     # Corner curve start
            [-1.9, 4.7],     # Corner curve
            [-2.0, 4.4],     # Corner curve end
            [-2.0, 2.4],     # Straight section
            [-2.0, 2.2],     # Corner curve start
            [-2.1, 2.1],     # Corner curve
            [-2.3, 2.0],     # Corner curve
            [-3.6, 2.0],     # Approach corner
            [-3.8, 1.9],     # Corner curve start
            [-3.9, 1.7],     # Corner curve
            [-4.0, 1.4],     # Corner curve end
            [-4.0, 0.4],     # Straight section
            [-4.0, 0.2],     # Corner curve start
            [-3.9, 0.1],     # Corner curve
            [-3.7, 0.0],     # Corner curve
            [-2.4, 0.0],     # Approach corner
            [-2.2, -0.1],    # Corner curve start
            [-2.1, -0.3],    # Corner curve
            [-2.0, -0.6],    # Corner curve end
            [-2.0, -1.6],    # Straight section
            [-2.0, -1.8],    # Corner curve start
            [-1.9, -1.9],    # Corner curve
            [-1.7, -2.0],    # Corner curve
            [0.6, -2.0],     # Approach corner
            [0.8, -1.9],     # Corner curve start
            [0.9, -1.7],     # Corner curve
            [1.0, -1.4],     # Corner curve end
            [1.0, 0.6],      # Straight section toward center
            [1.0, 0.8],      # Corner curve start
            [0.9, 0.9],      # Corner curve
            [0.7, 1.0],      # Corner curve
            [0.0, 1.0],      # Final approach
            [0.0, 0.0]       # Return to start
        ])
        
        # Timing: spend time proportional to segment length
        segment_lengths = np.linalg.norm(np.diff(waypoints, axis=0), axis=1)
        total_length = np.sum(segment_lengths)
        base_speed = 1.2  # m/s
        total_time = total_length / base_speed
        
        # Create cumulative time array
        segment_times = segment_lengths / base_speed
        cumulative_times = np.concatenate([[0], np.cumsum(segment_times)])
        
        # Normalize time to cycle period
        cycle_time = 60.0  # 60 seconds for full cycle
        t_normalized = (t % cycle_time) / cycle_time * total_time
        
        # Find current segment
        segment_idx = np.searchsorted(cumulative_times[1:], t_normalized)
        segment_idx = min(segment_idx, len(waypoints) - 2)
        
        # Interpolate position within segment
        t_start = cumulative_times[segment_idx]
        t_end = cumulative_times[segment_idx + 1]
        
        if t_end > t_start:
            alpha = (t_normalized - t_start) / (t_end - t_start)
        else:
            alpha = 0.0
        
        alpha = np.clip(alpha, 0.0, 1.0)
        
        # Current position
        p_start = waypoints[segment_idx]
        p_end = waypoints[segment_idx + 1]
        pos = p_start + alpha * (p_end - p_start)
        
        x_d, y_d = pos[0], pos[1]
        
        # Calculate desired heading (tangent to path)
        direction = p_end - p_start
        if np.linalg.norm(direction) > 1e-6:
            theta_d = np.arctan2(direction[1], direction[0])
        else:
            theta_d = 0.0
        
        # Feedforward velocities
        v_ff = base_speed
        
        # Calculate angular velocity for turns
        if segment_idx < len(waypoints) - 2:
            next_direction = waypoints[segment_idx + 2] - waypoints[segment_idx + 1]
            if np.linalg.norm(next_direction) > 1e-6:
                next_theta = np.arctan2(next_direction[1], next_direction[0])
                angle_diff = wrap_pi(next_theta - theta_d)
                # Smoother turns
                omega_ff = angle_diff * 0.3
            else:
                omega_ff = 0.0
        else:
            omega_ff = 0.0
            
        return np.array([x_d, y_d, theta_d]), v_ff, omega_ff
    else:  # línea
        v0 = 0.8
        x_d = v0 * t
        y_d = 0.0
        theta_d = 0.0
        return np.array([x_d, y_d, theta_d]), v0, 0.0

def wrap_pi(a):
    return np.arctan2(np.sin(a), np.cos(a))

def pose_error_body(x_est, x_ref):
    """Error de postura en marco del robot (unicycle).
       x_est=[x,y,theta], x_ref=[x_d,y_d,theta_d]"""
    dx = x_ref[0] - x_est[0]
    dy = x_ref[1] - x_est[1]
    c, s = np.cos(x_est[2]), np.sin(x_est[2])
    # transformar error a marco del robot
    e_x =  c*dx + s*dy
    e_y = -s*dx + c*dy
    e_theta = wrap_pi(x_ref[2] - x_est[2])
    # errores "PID": distancia (rho) y orientación (theta)
    rho = e_x  # aproximación "look-ahead": seguir el punto frente a la base
    theta_err = e_theta + np.arctan2(e_y, max(1e-6, abs(e_x))) * np.sign(e_x)
    return rho, theta_err


In [21]:
# === NEW: PID discretos para v y omega (con anti-windup y filtro derivativo) ===
class PID:
    def __init__(self, Kp, Ki, Kd, dt, umin=None, umax=None, tau_d=0.02):
        self.Kp, self.Ki, self.Kd = Kp, Ki, Kd
        self.dt = dt
        self.umin, self.umax = umin, umax
        self.tau_d = tau_d              # filtro derivativo (1ª orden)
        self.int = 0.0
        self.prev_e = 0.0
        self.d_filt = 0.0

    def reset(self):
        self.int = 0.0
        self.prev_e = 0.0
        self.d_filt = 0.0

    def step(self, e, u_ff=0.0):
        # derivada filtrada
        de = (e - self.prev_e) / self.dt
        alpha = self.dt / (self.tau_d + self.dt)
        self.d_filt = (1 - alpha) * self.d_filt + alpha * de

        # salida preliminar (sin integrar)
        u_p = self.Kp * e + self.Kd * self.d_filt + u_ff

        # anti-windup por back-calculation
        u_unsat = u_p + self.Ki * self.int
        u = u_unsat
        if self.umin is not None: u = max(self.umin, u)
        if self.umax is not None: u = min(self.umax, u)

        # back-calculation: integra sólo el error + corrección de saturación
        k_aw = 1.0
        self.int += (e + k_aw * (u - u_unsat)) * self.dt

        self.prev_e = e
        return u


In [22]:
# === NEW: controlador unicycle->diferencial con PID sobre (rho, theta_err) ===
class NeuralPIDController:
    def __init__(self, dt, L=0.5,
                 v_limits=(-1.5, 1.5), om_limits=(-2.5, 2.5),
                 gains_v=(1.2, 0.3, 0.05), gains_w=(2.0, 0.5, 5.0)):
        self.L = L
        self.pid_v = PID(*gains_v, dt=dt, umin=v_limits[0],  umax=v_limits[1],  tau_d=0.03)
        self.pid_w = PID(*gains_w, dt=dt, umin=om_limits[0], umax=om_limits[1], tau_d=0.03)

    def reset(self):
        self.pid_v.reset()
        self.pid_w.reset()

    def step(self, x_est, x_ref, v_ff=0.0, w_ff=0.0):
        rho, theta_err = pose_error_body(x_est, x_ref)
        v = self.pid_v.step(rho, u_ff=v_ff)
        w = self.pid_w.step(theta_err, u_ff=w_ff)

        # mapeo unicycle→diferencial
        v_l = v - 0.5 * self.L * w
        v_r = v + 0.5 * self.L * w

        # límites por rueda (opcionalmente distintos a los de v, ω)
        vmax_wheel = 2.0
        v_l = np.clip(v_l, -vmax_wheel, vmax_wheel)
        v_r = np.clip(v_r, -vmax_wheel, vmax_wheel)
        return np.array([v_l, v_r]), {'rho': rho, 'theta_err': theta_err, 'v_cmd': v, 'w_cmd': w}




In [23]:
# ============================================================
# 2) RHONN structure
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    # z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_x(x_est, u_input=None):
    """Features for predicting x(k+1)"""
    s_x = sigmoidal(x_est[0])
    s_y = sigmoidal(x_est[1])
    s_theta = sigmoidal(x_est[2])
    features = [
        s_x * s_y,
        s_x * s_theta,
        s_x**3,
        np.cos(x_est[2]),      # useful for x dynamics
        np.sin(x_est[2]),
        x_est[0],
        x_est[1],
        1.0
    ]
    if u_input is not None:
        s_vl = sigmoidal(u_input[0])
        s_vr = sigmoidal(u_input[1])
        features.extend([s_vl * s_vr, (u_input[0] + u_input[1]) / 2.0])  # avg linear vel
    else:
        features.extend([0.0, 0.0])
    return np.array(features)

def construct_z_y(x_est, u_input=None):
    """Features for predicting y(k+1)"""
    s_x = sigmoidal(x_est[0])
    s_y = sigmoidal(x_est[1])
    s_theta = sigmoidal(x_est[2])
    features = [
        s_x * s_y,
        s_y * s_theta,
        s_y**2,
        np.cos(x_est[2]),
        np.sin(x_est[2]),
        x_est[1],
        1.0
    ]
    if u_input is not None:
        s_vl = sigmoidal(u_input[0])
        s_vr = sigmoidal(u_input[1])
        features.extend([s_vl * s_vr, (u_input[0] + u_input[1]) / 2.0])
    else:
        features.extend([0.0, 0.0])
    return np.array(features)

def construct_z_theta(x_est, u_input=None):
    """Features for predicting theta(k+1)"""
    s_theta = sigmoidal(x_est[2])
    features = [
        s_theta,
        s_theta**2,
        x_est[2],
        1.0
    ]
    if u_input is not None:
        # Angular velocity: (v_r - v_l)/L
        L = 0.5
        omega = (u_input[1] - u_input[0]) / L
        features.extend([omega, sigmoidal(omega)])
    else:
        features.extend([0.0, 0.0])
    return np.array(features)


def RHONN_predict_state(state_idx, x_state_for_z, w_neuron, u_input=None):
    if state_idx == 0:   # x
        z = construct_z_x(x_state_for_z, u_input)
    elif state_idx == 1: # y
        z = construct_z_y(x_state_for_z, u_input)
    elif state_idx == 2: # theta
        z = construct_z_theta(x_state_for_z, u_input)
    else:
        raise ValueError("Only 3 states supported")
    
    if len(z) != len(w_neuron):
        raise ValueError(f"Feature/weight dim mismatch for state {state_idx}: z({len(z)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z)

In [24]:
# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    """
    EKF on each neuron's weight vector, with random-walk weight dynamics.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        
        # Handle both scalar and list for num_weights_per_neuron
        if isinstance(num_weights_per_neuron, (list, tuple)):
            self.num_weights_per_neuron = list(num_weights_per_neuron)
        else:
            self.num_weights_per_neuron = [num_weights_per_neuron] * num_neurons
            
        self.eta = eta

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            n_weights_i = self.num_weights_per_neuron[i]
            
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(n_weights_i) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(n_weights_i) * P_init)
            self.Q.append(np.eye(n_weights_i) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One EKF update for all neurons, using state-specific feature vectors.
        
        Args:
            chi_kp1: np.array, measured (noisy) states at time k+1  (target)
            chi_k  : np.array, filter's own estimate at time k     (used for z construction)
            x_hat_previous: np.array, previous estimate (time k)  (used for z construction)
            u_input: control input at time k (for mobile robot)
        """
        # No shared z vector — each neuron builds its own
        for i in range(self.num_neurons):
            n_weights_i = self.num_weights_per_neuron[i]
            
            # --- Build state-specific z_i ---
            if i == 0:      # x neuron
                z_i = construct_z_x(x_hat_previous, u_input)
            elif i == 1:    # y neuron
                z_i = construct_z_y(x_hat_previous, u_input)
            elif i == 2:    # theta neuron
                z_i = construct_z_theta(x_hat_previous, u_input)
            else:
                raise ValueError("Only 3 neurons (x, y, theta) supported.")

            # Ensure dimension consistency
            if len(z_i) != n_weights_i:
                raise ValueError(f"Neuron {i}: z vector length ({len(z_i)}) ≠ expected weights ({n_weights_i})")

            H_i = z_i.reshape(-1, 1)  # (n_w, 1)

            # --- EKF Prediction Step (random walk) ---
            P_pred = self.P[i] + self.Q[i]
            P_pred += np.eye(n_weights_i) * 1e-8  # regularization

            # --- Innovation ---
            x_hat_pred_i = np.dot(self.weights[i], z_i)  # scalar prediction
            e_i = chi_kp1[i] - x_hat_pred_i
            e_i = np.clip(e_i, -10.0, 10.0)  # prevent outlier-driven divergence

            # --- Innovation covariance ---
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-10:
                M_i = 1e-10

            # --- Kalman gain ---
            K_i = (P_pred @ H_i).flatten() / M_i  # (n_w,)

            # --- Adaptive learning rate (optional but recommended) ---
            adaptive_eta = self.eta * (1.0 / (1.0 + np.abs(e_i) * 0.1))

            # --- Weight update ---
            self.weights[i] += adaptive_eta * K_i * e_i

            # --- Covariance update (Joseph form for stability) ---
            I_KH = np.eye(n_weights_i) - np.outer(K_i, H_i.ravel())
            self.P[i] = I_KH @ P_pred @ I_KH.T + np.outer(K_i, K_i) * self.R[i][0]

            # --- Ensure symmetry & positive definiteness ---
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            min_eig = np.min(np.linalg.eigvals(self.P[i]))
            if min_eig <= 0:
                self.P[i] += np.eye(n_weights_i) * (1e-6 - min_eig + 1e-8)

In [25]:
# ============================================================
# 4) Particle Filter trainer over weights
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle filter over neuron weights (per neuron).
    - Predict (random walk on weights)
    - Update (likelihood from chi_{k+1} vs prediction built with z at k)
    - ESS-triggered systematic resampling (low variance, efficient)
    """
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=100,
                 initial_weights=None, Q_std=None, R_std=None, ess_threshold=None):
        self.num_neurons = num_neurons
        
        # Handle both scalar and list for num_weights_per_neuron
        if isinstance(num_weights_per_neuron, (list, tuple)):
            self.num_weights_per_neuron = list(num_weights_per_neuron)
        else:
            self.num_weights_per_neuron = [num_weights_per_neuron] * num_neurons
            
        self.n_particles = n_particles
        
        # Handle Q_std (process noise) - can be scalar or list/array per neuron
        if Q_std is None:
            self.Q_std = [0.05] * num_neurons
        elif isinstance(Q_std, (int, float)):
            self.Q_std = [Q_std] * num_neurons
        else:
            self.Q_std = list(Q_std) if len(Q_std) == num_neurons else [0.05] * num_neurons
            
        # Handle R_std (measurement noise) - can be scalar or list/array per neuron  
        if R_std is None:
            self.R_std = [0.1] * num_neurons
        elif isinstance(R_std, (int, float)):
            self.R_std = [R_std] * num_neurons
        else:
            self.R_std = list(R_std) if len(R_std) == num_neurons else [0.1] * num_neurons
            
        # Compute R_var for each neuron
        self.R_var = [r**2 for r in self.R_std]
        
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        # Initialize global weight estimates first
        self.weights = []
        if initial_weights is not None:
            self.weights = [np.copy(w) for w in initial_weights]
        else:
            self.weights = [np.random.randn(self.num_weights_per_neuron[i]) * 0.01 for i in range(num_neurons)]

        self.particles = []
        self.weights_pf = []

        for i in range(num_neurons):
            n_weights_i = self.num_weights_per_neuron[i]
            
            if initial_weights is not None and i < len(initial_weights):
                base = np.copy(initial_weights[i])
                # Adaptive initialization variance based on weight magnitudes
                weight_magnitude = np.std(base) if np.std(base) > 0 else 1.0
                init_std = max(0.01, min(0.1, weight_magnitude * 0.5))  # Adaptive but bounded
            else:
                base = self.weights[i]
                init_std = 0.05
            
            # Better initialization: base + controlled noise
            particles_i = base[np.newaxis, :] + np.random.randn(n_particles, n_weights_i) * init_std
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        """Effective Sample Size calculation with improved numerical stability."""
        w_norm = w / (np.sum(w) + 1e-100)
        return 1.0 / (np.sum(w_norm**2) + 1e-100)

    def _resample_systematic(self, neuron_index):
        """Systematic resampling (deterministic, low variance, more efficient than stratified)"""
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]

        w_norm = w / (np.sum(w) + 1e-100)
        N = len(w_norm)
        cdf = np.cumsum(w_norm)
        
        # Systematic positions: single random offset for all positions
        u = np.random.rand() / N  # Single random number in [0, 1/N)
        positions = u + np.arange(N) / N  # Equally spaced positions with random offset
        
        # Use searchsorted for efficient index finding (much faster than loops)
        indexes = np.searchsorted(cdf, positions)
        indexes = np.clip(indexes, 0, N - 1)  # Handle edge cases

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One PF step over all neuron weight-sets using state-specific feature vectors.
        
        Args:
            chi_kp1: measured true states at k+1 (targets)
            chi_k  : filter's own estimate at k   (for z construction)
            x_hat_previous: previous estimate at k (to complete z)
            u_input: control input at time k (for mobile robot)
        """
        # 1) Predict: random walk on weights with state-specific Q_std
        for i in range(self.num_neurons):
            n_weights_i = self.num_weights_per_neuron[i]
            self.particles[i] += np.random.randn(self.n_particles, n_weights_i) * self.Q_std[i]
        
        # 2) Update: compute importance weights using state-specific z_i
        for i in range(self.num_neurons):
            n_weights_i = self.num_weights_per_neuron[i]
            
            # --- Build state-specific z_i for neuron i ---
            if i == 0:      # x neuron
                z_i = construct_z_x(x_hat_previous, u_input)
            elif i == 1:    # y neuron
                z_i = construct_z_y(x_hat_previous, u_input)
            elif i == 2:    # theta neuron
                z_i = construct_z_theta(x_hat_previous, u_input)
            else:
                raise ValueError("Only 3 neurons (x, y, theta) supported.")
            
            if len(z_i) != n_weights_i:
                raise ValueError(f"Neuron {i}: z length ({len(z_i)}) ≠ expected weights ({n_weights_i})")

            # Compute predictions for all particles
            w_mat = self.particles[i]                # (N, num_weights)
            x_pred_particles = w_mat @ z_i           # (N,) — RHONN output for each particle

            # Compute innovation
            innov = chi_kp1[i] - x_pred_particles    # (N,)

            # Robust Gaussian likelihood (avoid underflow)
            var_robust = max(self.R_var[i], 1e-6)
            ll = -0.5 * (innov**2) / var_robust - 0.5 * np.log(2 * np.pi * var_robust)

            # Numerical stability: subtract max log-likelihood
            ll_max = np.max(ll)
            ll_normalized = ll - ll_max
            like = np.exp(np.clip(ll_normalized, -20, 0))  # Prevent underflow

            # Update particle weights
            self.weights_pf[i] *= (like + 1e-15)
            w_sum = np.sum(self.weights_pf[i])
            if w_sum < 1e-15:
                # Complete weight collapse → reset uniformly
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= w_sum

            # 3) Resample if Effective Sample Size is too low
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)

        # 4) Update global weight estimates (for prediction and logging)
        if not hasattr(self, 'weights') or len(self.weights) != self.num_neurons:
            self.weights = [np.zeros(self.num_weights_per_neuron) for _ in range(self.num_neurons)]
        
        for i in range(self.num_neurons):
            w_norm = self.weights_pf[i] / (np.sum(self.weights_pf[i]) + 1e-15)
            self.weights[i] = np.sum(w_norm[:, np.newaxis] * self.particles[i], axis=0)

    def get_estimate(self):
        """Return current weight estimates (maintained consistently with particles)."""
        if hasattr(self, 'weights') and len(self.weights) == self.num_neurons:
            return self.weights
        else:
            # Fallback to simple mean if weights not properly maintained
            return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]

    def get_parameters_info(self):
        """Return comprehensive information about the PF parameters and state for each neuron."""
        state_names = ['x', 'y', 'theta']
        info = {}
        for i in range(self.num_neurons):
            name = state_names[i] if i < len(state_names) else f'state_{i}'
            current_ess = self._ess(self.weights_pf[i]) if hasattr(self, 'weights_pf') else 'N/A'
            info[name] = {
                'Q_std': self.Q_std[i],
                'R_std': self.R_std[i], 
                'R_var': self.R_var[i],
                'n_particles': self.n_particles,
                'ess_threshold': self.ess_threshold,
                'current_ess': current_ess,
                'ess_ratio': current_ess / self.n_particles if isinstance(current_ess, (int, float)) else 'N/A'
            }
        return info

In [26]:
# ============================================================
# 4b) Unscented Kalman Filter (UKF) trainer over weights
# ============================================================
class UKF_RHONN_Trainer:
    """
    Unscented Kalman Filter on each neuron's weight vector.
    Uses sigma points to handle nonlinearities better than standard EKF.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0, 
                 alpha=1e-3, beta=2.0, kappa=None):
        self.num_neurons = num_neurons
        
        # Handle both scalar and list for num_weights_per_neuron
        if isinstance(num_weights_per_neuron, (list, tuple)):
            self.num_weights_per_neuron = list(num_weights_per_neuron)
        else:
            self.num_weights_per_neuron = [num_weights_per_neuron] * num_neurons
            
        self.eta = eta
        
        # UKF parameters
        self.alpha = alpha  # Spread of sigma points (typically 1e-4 to 1)
        self.beta = beta    # Prior knowledge about distribution (2 for Gaussian)
        
        # Store UKF parameters per neuron (different dimensions)
        self.kappa = []
        self.lambda_ = []
        self.Wm = []  # Mean weights for sigma points per neuron
        self.Wc = []  # Covariance weights for sigma points per neuron
        
        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            n_weights_i = self.num_weights_per_neuron[i]
            
            # UKF parameters for this neuron
            kappa_i = kappa if kappa is not None else 3 - n_weights_i
            lambda_i = alpha**2 * (n_weights_i + kappa_i) - n_weights_i
            
            self.kappa.append(kappa_i)
            self.lambda_.append(lambda_i)
            
            # Weights for mean and covariance computation for this neuron
            Wm_i = np.zeros(2 * n_weights_i + 1)
            Wc_i = np.zeros(2 * n_weights_i + 1)
            
            Wm_i[0] = lambda_i / (n_weights_i + lambda_i)
            Wc_i[0] = lambda_i / (n_weights_i + lambda_i) + (1 - alpha**2 + beta)
            
            for j in range(1, 2 * n_weights_i + 1):
                Wm_i[j] = 1.0 / (2 * (n_weights_i + lambda_i))
                Wc_i[j] = 1.0 / (2 * (n_weights_i + lambda_i))
            
            self.Wm.append(Wm_i)
            self.Wc.append(Wc_i)
            
            # Initialize weights for this neuron
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(n_weights_i) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(n_weights_i) * P_init)
            self.Q.append(np.eye(n_weights_i) * Q_init)
            self.R.append(np.array([R_init]))

    def _generate_sigma_points(self, mean, covariance, neuron_id):
        """Generate sigma points for UKF for specific neuron."""
        n = len(mean)
        sigma_points = np.zeros((2 * n + 1, n))
        lambda_i = self.lambda_[neuron_id]
        
        # First sigma point is the mean
        sigma_points[0] = mean
        
        # Calculate matrix square root
        try:
            sqrt = np.linalg.cholesky((n + lambda_i) * covariance)
        except np.linalg.LinAlgError:
            # If Cholesky fails, use SVD
            U, s, Vh = np.linalg.svd(covariance)
            sqrt = U @ np.diag(np.sqrt(s)) @ Vh
            sqrt *= np.sqrt(n + lambda_i)
        
        # Generate remaining sigma points
        for i in range(n):
            sigma_points[i + 1] = mean + sqrt[i]
            sigma_points[i + 1 + n] = mean - sqrt[i]
        
        return sigma_points

    def _measurement_function(self, weight_sigma_point, z_vector):
        """Measurement function: applies RHONN prediction with given weights."""
        return np.dot(weight_sigma_point, z_vector)

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One UKF update for all neurons using state-specific feature vectors.
        
        Args:
            chi_kp1: np.array, measured states at time k+1 (target)
            chi_k  : np.array, filter's own estimate at time k (used in z construction)
            x_hat_previous: np.array, previous state estimate (time k) for z
            u_input: control input [v_l, v_r] at time k
        """
        for i in range(self.num_neurons):
            n_weights_i = self.num_weights_per_neuron[i]
            
            # --- Build state-specific z_i ---
            if i == 0:      # x neuron
                z_i = construct_z_x(x_hat_previous, u_input)
            elif i == 1:    # y neuron
                z_i = construct_z_y(x_hat_previous, u_input)
            elif i == 2:    # theta neuron
                z_i = construct_z_theta(x_hat_previous, u_input)
            else:
                raise ValueError("Only 3 neurons (x, y, theta) supported.")

            if len(z_i) != n_weights_i:
                raise ValueError(f"Neuron {i}: z length ({len(z_i)}) ≠ expected weights ({n_weights_i})")

            # --- Prediction Step ---
            sigma_points = self._generate_sigma_points(self.weights[i], self.P[i], i)
            predicted_sigma_points = sigma_points.copy()  # weights unchanged in prediction (random walk assumed)

            # Predict mean and covariance using neuron-specific weights
            predicted_mean = np.sum(self.Wm[i][:, np.newaxis] * predicted_sigma_points, axis=0)
            predicted_cov = self.Q[i].copy()
            for j in range(2 * n_weights_i + 1):
                diff = predicted_sigma_points[j] - predicted_mean
                predicted_cov += self.Wc[i][j] * np.outer(diff, diff)

            # --- Measurement Update ---
            # Transform sigma points through RHONN measurement function
            measurement_sigma_points = np.array([
                self._measurement_function(predicted_sigma_points[j], z_i)
                for j in range(2 * n_weights_i + 1)
            ])

            # Predicted measurement (scalar)
            predicted_measurement = np.sum(self.Wm[i] * measurement_sigma_points)

            # Innovation covariance
            innovation_cov = self.R[i][0]
            for j in range(2 * n_weights_i + 1):
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                innovation_cov += self.Wc[i][j] * (diff_meas ** 2)
            if innovation_cov < 1e-12:
                innovation_cov = 1e-12

            # Cross-covariance
            cross_cov = np.zeros(n_weights_i)
            for j in range(2 * n_weights_i + 1):
                diff_state = predicted_sigma_points[j] - predicted_mean
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                cross_cov += self.Wc[i][j] * diff_state * diff_meas

            # Kalman gain
            K = cross_cov / innovation_cov

            # Innovation
            innovation = chi_kp1[i] - predicted_measurement
            innovation = np.clip(innovation, -10.0, 10.0)  # optional: prevent outliers

            # State (weight) update with adaptive learning rate
            adaptive_eta = self.eta * (1.0 / (1.0 + np.abs(innovation) * 0.1))
            self.weights[i] = predicted_mean + adaptive_eta * K * innovation

            # Covariance update
            self.P[i] = predicted_cov - np.outer(K, K) * innovation_cov

            # Ensure symmetry and positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            min_eig = np.min(np.linalg.eigvals(self.P[i]))
            if min_eig <= 0:
                self.P[i] += np.eye(n_weights_i) * (1e-6 - min_eig + 1e-8)

In [27]:
# ============================================================
# Particle Swarm Optimization (PSO) Optimizer (lightweight)
# This replaces the previous differential_evolution implementation but keeps the same
# function signature and return structure so existing call sites don't need changes.
# ============================================================

def differential_evolution(objective, bounds, pop_factor=10, F=0.7, CR=0.9, generations=30, seed=None, tol=1e-6, stall_generations=8):
    """Lightweight PSO exposed under the name `differential_evolution` for API compatibility.
    Parameters:
      objective: callable(x) -> float (minimized)
      bounds: list of (low, high) pairs for each dimension
      pop_factor, F, CR: kept for compatibility but different meaning here
      generations: number of PSO iterations
      seed: RNG seed
    Returns dict with keys: best_params (ndarray), best_score (float), history (list of (iter,score))"""
    if seed is not None:
        np.random.seed(seed)
    dim = len(bounds)
    # swarm size scaled similarly to previous pop_size heuristic
    swarm_size = max(int(pop_factor * dim), 8)
    # PSO hyperparams (some mapped from DE args for convenience)
    w = 0.7  # inertia
    c1 = 1.5  # cognitive
    c2 = 1.5  # social

    # Initialize particles uniformly inside bounds
    lb = np.array([b[0] for b in bounds])
    ub = np.array([b[1] for b in bounds])
    pos = lb + (ub - lb) * np.random.rand(swarm_size, dim)
    vel = (ub - lb) * (np.random.rand(swarm_size, dim) - 0.5) * 0.1
    scores = np.array([objective(p) for p in pos])
    pbest_pos = pos.copy()
    pbest_scores = scores.copy()
    gbest_idx = int(np.argmin(pbest_scores))
    gbest_pos = pbest_pos[gbest_idx].copy()
    gbest_score = float(pbest_scores[gbest_idx])
    history = [(0, gbest_score)]
    no_improve = 0

    for it in range(1, generations+1):
        r1 = np.random.rand(swarm_size, dim)
        r2 = np.random.rand(swarm_size, dim)
        vel = w * vel + c1 * r1 * (pbest_pos - pos) + c2 * r2 * (gbest_pos - pos)
        pos = pos + vel
        # clamp
        pos = np.maximum(pos, lb)
        pos = np.minimum(pos, ub)
        # evaluate
        for i in range(swarm_size):
            try:
                s = objective(pos[i])
            except Exception as e:
                # if objective fails, treat as very bad score
                s = float('inf')
            scores[i] = s
            if s < pbest_scores[i] - tol:
                pbest_scores[i] = s
                pbest_pos[i] = pos[i].copy()
                if s < gbest_score - tol:
                    gbest_score = s
                    gbest_pos = pos[i].copy()
        history.append((it, float(gbest_score)))
        if history[-1][1] < history[-2][1] - tol:
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= stall_generations:
            break
    return {'best_params': gbest_pos, 'best_score': gbest_score, 'history': history}

In [28]:
# ============================================================
# 5) Simulation Main Loop (uses optimized params if present)
# ============================================================

# --- Simulation settings ---
n_steps = 3000  # Increased for city streets (60s cycle)
dt = 0.02
t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

process_noise_type = 'laplacian'  # 'mixed' | 'laplacian' | 'uniform' | 'gaussian'
process_noise_std = 0.0
terrain_roughness = 0.0
sensor_bias = [0.0, 0.0, 0.0]  # Small systematic biases [x, y, theta]

# --- Control settings ---
control_enabled = True
ref_kind = 'city_streets'  # 'circle' | 'lemniscate' | 'line' | 'city_streets'
controller = NeuralPIDController(dt=dt, L=0.5, 
                                gains_v=(3.0, 0.08, 0.08),     # Kp, Ki, Kd for velocity
                                gains_w=(30.0, 0.09, 1.5))     # Kp, Ki, Kd for angular velocity



# --- True system init ---
x_true = np.zeros((n_steps, 3))
x_true[0] = [0.0, 0.0, 0.0]  # Initial conditions for mobile robot [x, y, theta]



# --- RHONN config with state-specific regressors ---
num_neurons = 3  # Three states for mobile robot [x, y, theta]

# Determine regressor dimensions by testing
test_state = np.array([0.0, 0.0, 0.0])
test_control = np.array([0.0, 0.0])

num_weights_per_neuron = [
    len(construct_z_x(test_state, test_control)),      # x-state: 10 features
    len(construct_z_y(test_state, test_control)),      # y-state: 9 features  
    len(construct_z_theta(test_state, test_control))   # theta-state: 6 features
]

print(f"State-specific regressor dimensions: {num_weights_per_neuron}")

# --- State-specific initial weights ---
np.random.seed(42)  # For reproducible results
common_initial_weights = [
    np.random.uniform(-0.5, 0.5, num_weights_per_neuron[0]),  # x-neuron (10 weights)
    np.random.uniform(-0.5, 0.5, num_weights_per_neuron[1]),  # y-neuron (9 weights)
    np.random.uniform(-0.5, 0.5, num_weights_per_neuron[2])   # theta-neuron (6 weights)
]

print("State-Specific Initial Weights:")
state_names = ['x', 'y', 'theta']
for i, w in enumerate(common_initial_weights):
    print(f"  {state_names[i]}-neuron ({len(w)} weights): {w}")

# Extract optimized parameters if available
opt_EKF = optimized_params.get('EKF') if 'optimized_params' in globals() else None
opt_UKF = optimized_params.get('UKF') if 'optimized_params' in globals() else None
opt_PF  = optimized_params.get('PF')  if 'optimized_params' in globals() else None

# Fallback defaults
if opt_EKF is None:
    opt_EKF = [2e-4, 8e-3, 1.5, 0.4]
if opt_UKF is None:
    opt_UKF = [2e-4, 8e-3, 1.5, 0.6, 1e-2]


# --- EKF ---
ekf_trainer = EKF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    initial_weights=common_initial_weights,
    Q_init=opt_EKF[0], R_init=opt_EKF[1], P_init=opt_EKF[2], eta=opt_EKF[3]
)
x_hat_ekf = np.zeros((n_steps, 3))
x_hat_ekf[0] = x_true[0]

# --- UKF ---
ukf_trainer = UKF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    initial_weights=common_initial_weights,
    Q_init=opt_UKF[0], R_init=opt_UKF[1], P_init=opt_UKF[2], eta=opt_UKF[3],
    alpha=opt_UKF[4], beta=2.0
)
x_hat_ukf = np.zeros((n_steps, 3))
x_hat_ukf[0] = x_true[0]

# --- PF --- (n_particles fixed at 800)
n_particles = 800

Q_std_per_state = [0.1, 0.1, 0.5]  # Process noise: x, y (same), theta (smaller)
R_std_per_state = [0.1, 0.1, 0.5]  # Measurement noise: x, y (same), theta (larger)

ess_threshold = n_particles * 0.5  # Half of particles

pf_trainer = PF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    n_particles=n_particles,
    initial_weights=common_initial_weights,
    Q_std=Q_std_per_state, R_std=R_std_per_state,
    ess_threshold=ess_threshold
)

x_hat_pf = np.zeros((n_steps, 3))
x_hat_pf[0] = x_true[0]

def get_estimate_for_control(k):
    """Select which RHONN estimate to use for control feedback: PF/EKF/UKF"""
    use = 'PF'  # Change to 'EKF' or 'UKF' if desired
    if use == 'PF':
        return x_hat_pf[k]
    elif use == 'EKF':
        return x_hat_ekf[k]
    else:
        return x_hat_ukf[k]
    

# --- Main simulation loop ---

print("\nStarting mobile robot simulation (optimized params)...")
for k in range(n_steps - 1):
    t_current = k * dt
    # Control plant using RHONN feedback
    if control_enabled:
        x_ref_k, v_ff, w_ff = reference_trajectory(t_current, ref_kind)
        x_est_k = get_estimate_for_control(k)  # RHONN estimate (PF default)
        u_current, ctrl_dbg = controller.step(x_est_k, x_ref_k, v_ff=v_ff, w_ff=w_ff)
    x_true[k+1] = plant(x_true[k], u_current, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)

    # --- Measurement noise injection (simple measurement model) ---
    # Simulate GPS + compass noise on x, y, theta
    meas_noise_std = np.array([0.0, 0.0, 0.0])  # adjust as needed
    meas_bias = np.array([0.0, 0.0, 0.0])          # you can make this nonzero later
    y_kp1 = x_true[k+1] + np.random.normal(meas_bias, meas_noise_std)
    x_true[k+1] = y_kp1  # For simplicity, use noisy measurement as "true" for filter updates

    # EKF
    ekf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_ekf[k], x_hat_previous=x_hat_ekf[k], u_input=u_current)
    x_state_for_z_ekf = np.copy(x_hat_ekf[k])
    x_state_for_z_ekf[0] = x_hat_ekf[k][0]
    x_hat_ekf[k+1, 0] = RHONN_predict_state(0, x_hat_ekf[k], ekf_trainer.weights[0], u_current)
    x_hat_ekf[k+1, 1] = RHONN_predict_state(1, x_hat_ekf[k], ekf_trainer.weights[1], u_current)
    x_hat_ekf[k+1, 2] = RHONN_predict_state(2, x_hat_ekf[k], ekf_trainer.weights[2], u_current)

    # UKF
    ukf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_ukf[k], x_hat_previous=x_hat_ukf[k], u_input=u_current)
    x_state_for_z_ukf = np.copy(x_hat_ukf[k])
    x_state_for_z_ukf[0] = x_hat_ukf[k][0]
    x_hat_ukf[k+1, 0] = RHONN_predict_state(0, x_hat_ukf[k], ukf_trainer.weights[0], u_current)
    x_hat_ukf[k+1, 1] = RHONN_predict_state(1, x_hat_ukf[k], ukf_trainer.weights[1], u_current)
    x_hat_ukf[k+1, 2] = RHONN_predict_state(2, x_hat_ukf[k], ukf_trainer.weights[2], u_current)

    # PF
    pf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_pf[k], x_hat_previous=x_hat_pf[k], u_input=u_current)
    pf_weight_estimates = pf_trainer.get_estimate()
    x_state_for_z_pf = np.copy(x_hat_pf[k])
    x_state_for_z_pf[0] = x_hat_pf[k][0]
    x_hat_pf[k+1, 0] = RHONN_predict_state(0, x_hat_pf[k], pf_weight_estimates[0], u_current)
    x_hat_pf[k+1, 1] = RHONN_predict_state(1, x_hat_pf[k], pf_weight_estimates[1], u_current)
    x_hat_pf[k+1, 2] = RHONN_predict_state(2, x_hat_pf[k], pf_weight_estimates[2], u_current)

    if k % (n_steps // 10) == 0:
        print(f"Simulation progress: {k/n_steps*100:.1f}%")

print("Simulation finished.")

State-specific regressor dimensions: [10, 9, 6]
State-Specific Initial Weights:
  x-neuron (10 weights): [-0.12545988  0.45071431  0.23199394  0.09865848 -0.34398136 -0.34400548
 -0.44191639  0.36617615  0.10111501  0.20807258]
  y-neuron (9 weights): [-0.47941551  0.46990985  0.33244264 -0.28766089 -0.31817503 -0.31659549
 -0.19575776  0.02475643 -0.06805498]
  theta-neuron (6 weights): [-0.20877086  0.11185289 -0.36050614 -0.20785535 -0.13363816 -0.04393002]

Starting mobile robot simulation (optimized params)...
Simulation progress: 0.0%
Simulation progress: 10.0%
Simulation progress: 10.0%
Simulation progress: 20.0%
Simulation progress: 20.0%
Simulation progress: 30.0%
Simulation progress: 30.0%
Simulation progress: 40.0%
Simulation progress: 40.0%
Simulation progress: 50.0%
Simulation progress: 50.0%
Simulation progress: 60.0%
Simulation progress: 60.0%
Simulation progress: 70.0%
Simulation progress: 70.0%
Simulation progress: 80.0%
Simulation progress: 80.0%
Simulation progress: 

In [29]:
# ============================================================
# 6) Results & plots for Differential Drive Mobile Robot
# ============================================================

# compute RMSE per state and per filter
rmse_x_ekf = np.sqrt(np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2))
rmse_y_ekf = np.sqrt(np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2))
rmse_theta_ekf = np.sqrt(np.mean((x_true[:, 2] - x_hat_ekf[:, 2])**2))

rmse_x_ukf = np.sqrt(np.mean((x_true[:, 0] - x_hat_ukf[:, 0])**2))
rmse_y_ukf = np.sqrt(np.mean((x_true[:, 1] - x_hat_ukf[:, 1])**2))
rmse_theta_ukf = np.sqrt(np.mean((x_true[:, 2] - x_hat_ukf[:, 2])**2))

rmse_x_pf = np.sqrt(np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2))
rmse_y_pf = np.sqrt(np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2))
rmse_theta_pf = np.sqrt(np.mean((x_true[:, 2] - x_hat_pf[:, 2])**2))

# total RMSE (combined across states)
rmse_total_ekf = np.sqrt(rmse_x_ekf**2 + rmse_y_ekf**2 + rmse_theta_ekf**2)
rmse_total_ukf = np.sqrt(rmse_x_ukf**2 + rmse_y_ukf**2 + rmse_theta_ukf**2)
rmse_total_pf = np.sqrt(rmse_x_pf**2 + rmse_y_pf**2 + rmse_theta_pf**2)
rmse_totals = {'EKF': rmse_total_ekf, 'UKF': rmse_total_ukf, 'PF': rmse_total_pf}
best_filter = min(rmse_totals, key=rmse_totals.get)


print("🎯" + "="*65)
print(f"🏆 MEJOR FILTRO: {best_filter} (RMSE total: {rmse_totals[best_filter]:.6f})")
print(f"🎲 SEMILLA USADA: {RANDOM_SEED}")
print("="*67)


print(f"\nFinal EKF-RHONN Weights:")
for i in range(3):
    state_names = ['x', 'y', 'theta']
    print(f"  Neuron {i+1} ({state_names[i]}): {ekf_trainer.weights[i]}")

print(f"\nFinal UKF-RHONN Weights:")
for i in range(3):
    state_names = ['x', 'y', 'theta']
    print(f"  Neuron {i+1} ({state_names[i]}): {ukf_trainer.weights[i]}")

print(f"\nFinal PF-RHONN Weight Estimates:")
pf_estimates = pf_trainer.get_estimate()
for i in range(3):
    state_names = ['x', 'y', 'theta']
    print(f"  Neuron {i+1} ({state_names[i]}): {pf_estimates[i]}")

print("\n--- Performance Comparison (RMSE) ---")
print(f"EKF RMSE x:     {rmse_x_ekf:.6f}")
print(f"EKF RMSE y:     {rmse_y_ekf:.6f}")
print(f"EKF RMSE theta: {rmse_theta_ekf:.6f}")
print(f"UKF RMSE x:     {rmse_x_ukf:.6f}")
print(f"UKF RMSE y:     {rmse_y_ukf:.6f}")
print(f"UKF RMSE theta: {rmse_theta_ukf:.6f}")
print(f"PF  RMSE x:     {rmse_x_pf:.6f}")
print(f"PF  RMSE y:     {rmse_y_pf:.6f}")
print(f"PF  RMSE theta: {rmse_theta_pf:.6f}")

states_info = [
    {'idx': 0, 'var': 'x', 'desc': 'X Position', 'y_label': 'X Position (m)', 'chi': 'χₓ (True X)', 'x': 'X (Est.)'},
    {'idx': 1, 'var': 'y', 'desc': 'Y Position', 'y_label': 'Y Position (m)', 'chi': 'χᵧ (True Y)', 'x': 'Y (Est.)'},
    {'idx': 2, 'var': 'theta', 'desc': 'Orientation', 'y_label': 'Orientation (rad)', 'chi': 'χθ (True θ)', 'x': 'θ (Est.)'}
]

for state_info in states_info:
    i = state_info['idx']
    trace_plant = go.Scatter(x=t_history, y=x_true[:, i], mode='lines', name=state_info['chi'], line=dict(color='black', width=2))
    trace_ekf = go.Scatter(x=t_history, y=x_hat_ekf[:, i], mode='lines', name=f"{state_info['x']} (EKF)", line=dict(dash='dash', color='blue'))
    trace_ukf = go.Scatter(x=t_history, y=x_hat_ukf[:, i], mode='lines', name=f"{state_info['x']} (UKF)", line=dict(dash='dashdot', color='green'))
    trace_pf = go.Scatter(x=t_history, y=x_hat_pf[:, i], mode='lines', name=f"{state_info['x']} (PF)", line=dict(dash='dot', color='red'))
    fig = go.Figure([trace_plant, trace_ekf, trace_ukf, trace_pf])
    fig.update_layout(title=f'Mobile Robot RHONN Identification - {state_info["var"]}', xaxis_title='Time (s)', yaxis_title=state_info['y_label'], legend=dict(x=0, y=1, orientation='h'), font=dict(size=12), plot_bgcolor='white', paper_bgcolor='white')
    fig.show()

error_x_ekf = x_true[:, 0] - x_hat_ekf[:, 0]
error_y_ekf = x_true[:, 1] - x_hat_ekf[:, 1]
error_theta_ekf = x_true[:, 2] - x_hat_ekf[:, 2]
error_x_ukf = x_true[:, 0] - x_hat_ukf[:, 0]
error_y_ukf = x_true[:, 1] - x_hat_ukf[:, 1]
error_theta_ukf = x_true[:, 2] - x_hat_ukf[:, 2]
error_x_pf = x_true[:, 0] - x_hat_pf[:, 0]
error_y_pf = x_true[:, 1] - x_hat_pf[:, 1]
error_theta_pf = x_true[:, 2] - x_hat_pf[:, 2]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=t_history, y=error_x_ekf, mode='lines', name=f'EKF Err X ({rmse_x_ekf:.2e})', opacity=0.7, line=dict(color='blue')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x_ukf, mode='lines', name=f'UKF Err X ({rmse_x_ukf:.2e})', opacity=0.7, line=dict(color='green')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x_pf, mode='lines', name=f'PF Err X ({rmse_x_pf:.2e})', opacity=0.7, line=dict(color='red')))
fig2.add_trace(go.Scatter(x=t_history, y=error_y_ekf, mode='lines', name=f'EKF Err Y ({rmse_y_ekf:.2e})', opacity=0.7, line=dict(color='blue', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_y_ukf, mode='lines', name=f'UKF Err Y ({rmse_y_ukf:.2e})', opacity=0.7, line=dict(color='green', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_y_pf, mode='lines', name=f'PF Err Y ({rmse_y_pf:.2e})', opacity=0.7, line=dict(color='red', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_ekf, mode='lines', name=f'EKF Err θ ({rmse_theta_ekf:.2e})', opacity=0.7, line=dict(color='blue', dash='dash')))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_ukf, mode='lines', name=f'UKF Err θ ({rmse_theta_ukf:.2e})', opacity=0.7, line=dict(color='green', dash='dash')))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_pf, mode='lines', name=f'PF Err θ ({rmse_theta_pf:.2e})', opacity=0.7, line=dict(color='red', dash='dash')))
fig2.update_layout(title='Identification Errors (RMSE values)', xaxis_title='Time (s)', yaxis_title='Error', legend=dict(x=0, y=1, orientation='h'), font=dict(size=12), plot_bgcolor='white', paper_bgcolor='white')
fig2.show()

# 2D Trajectory plot with orientation arrows
fig_trajectory = go.Figure()
fig_trajectory.add_trace(go.Scatter(x=x_true[:, 0], y=x_true[:, 1], mode='lines', name='True', line=dict(color='black', width=3)))
fig_trajectory.add_trace(go.Scatter(x=x_hat_ekf[:, 0], y=x_hat_ekf[:, 1], mode='lines', name='EKF', line=dict(color='blue', width=2, dash='dash')))
fig_trajectory.add_trace(go.Scatter(x=x_hat_ukf[:, 0], y=x_hat_ukf[:, 1], mode='lines', name='UKF', line=dict(color='green', width=2, dash='dashdot')))
fig_trajectory.add_trace(go.Scatter(x=x_hat_pf[:, 0], y=x_hat_pf[:, 1], mode='lines', name='PF', line=dict(color='red', width=2, dash='dot')))
fig_trajectory.add_trace(go.Scatter(x=[x_true[0, 0]], y=[x_true[0, 1]], mode='markers', name='Start', marker=dict(color='green', size=10, symbol='star')))
fig_trajectory.add_trace(go.Scatter(x=[x_true[-1, 0]], y=[x_true[-1, 1]], mode='markers', name='End', marker=dict(color='red', size=10, symbol='square')))

# Add orientation arrows for the true trajectory
arrow_step = max(1, n_steps // 30)  # Show ~30 arrows total
arrow_length = 0.3  # Arrow length in meters
for i in range(0, n_steps, arrow_step):
    x_pos = x_true[i, 0]
    y_pos = x_true[i, 1]
    theta = x_true[i, 2]
    
    # Calculate arrow end point
    x_end = x_pos + arrow_length * np.cos(theta)
    y_end = y_pos + arrow_length * np.sin(theta)
    
    # Add arrow as annotation
    fig_trajectory.add_annotation(
        x=x_end, y=y_end,
        ax=x_pos, ay=y_pos,
        xref='x', yref='y',
        axref='x', ayref='y',
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=2,
        arrowcolor='darkred',
        opacity=0.7
    )

# Add orientation arrows for the best performing filter (PF)
for i in range(0, n_steps, arrow_step * 2):  # Fewer arrows to avoid clutter
    x_pos = x_hat_pf[i, 0]
    y_pos = x_hat_pf[i, 1]
    theta = x_hat_pf[i, 2]
    
    # Calculate arrow end point (slightly shorter)
    x_end = x_pos + (arrow_length * 0.7) * np.cos(theta)
    y_end = y_pos + (arrow_length * 0.7) * np.sin(theta)
    
    # Add arrow as annotation
    fig_trajectory.add_annotation(
        x=x_end, y=y_end,
        ax=x_pos, ay=y_pos,
        xref='x', yref='y',
        axref='x', ayref='y',
        showarrow=True,
        arrowhead=1,
        arrowsize=0.8,
        arrowwidth=1.5,
        arrowcolor='red',
        opacity=0.5
    )

fig_trajectory.update_layout(
    title='Trajectory Comparison with Orientation Arrows (Optimized Params)', 
    xaxis_title='X (m)', 
    yaxis_title='Y (m)', 
    font=dict(size=12), 
    plot_bgcolor='white', 
    paper_bgcolor='white', 
    showlegend=True,
    # Ensure equal aspect ratio for proper visualization
    yaxis=dict(scaleanchor="x", scaleratio=1)
)
fig_trajectory.show()


# Performance Summary
print("\n📊 RMSE by Filter:")
print(f"   EKF: {rmse_total_ekf:.6f}  |  UKF: {rmse_total_ukf:.6f}  |  PF: {rmse_total_pf:.6f}")
print(f"\n💡 To reproduce these results:")
print(f"   Main: RANDOM_SEED = {RANDOM_SEED}")
print(f"   DE Optim: DE_SEED = {(RANDOM_SEED + 12345) % 100000}")

# --- Parameter summary ---
print("\n--- Optimized Parameter Summary ---")
print(f"EKF params: Q={ekf_trainer.Q[0][0,0]:.3e} R={ekf_trainer.R[0][0]:.3e} P0~{ekf_trainer.P[0][0,0]:.3e} eta={ekf_trainer.eta:.3f}")
print(f"UKF params: alpha={ukf_trainer.alpha:.3e} eta={ukf_trainer.eta:.3f} Qdiag={ukf_trainer.Q[0][0,0]:.3e} R={ukf_trainer.R[0][0]:.3e}")
print(f"PF params: Q_std={pf_trainer.Q_std} R_std={pf_trainer.R_std} ESS_th={pf_trainer.ess_threshold:.1f} n_particles={pf_trainer.n_particles}")

print("\nOptimization + simulation complete.")

🎯=================================================================
🏆 MEJOR FILTRO: PF (RMSE total: 0.136737)
🎲 SEMILLA USADA: 59034

Final EKF-RHONN Weights:
  Neuron 1 (x): [-0.29308834 -0.00227438  0.46725747  0.0065466  -0.00945558  0.92908404
  0.02734421  0.01771849  0.10997331 -0.07182517]
  Neuron 2 (y): [ 0.2333072  -0.0013754   0.10285744 -0.00226792 -0.00263101  0.94503958
 -0.07798104 -0.05107883  0.00441445]
  Neuron 3 (theta): [-2.2161858   3.07216245  0.30764733 -0.56586691  0.0196196   0.02344122]

Final UKF-RHONN Weights:
  Neuron 1 (x): [ 0.16008074 -0.9786985   0.5282576  -0.06024228 -0.04047667  0.36797876
 -0.21749681  0.01051523  0.05649127 -0.09495535]
  Neuron 2 (y): [ 0.73896778 -0.51404395  0.77465424 -0.2108251  -0.11502534  0.7091544
 -0.38124239  0.13149079 -0.04273943]
  Neuron 3 (theta): [ 1.49263773  0.76368683  0.1138628  -1.15866928  0.12086472 -0.61555221]

Final PF-RHONN Weight Estimates:
  Neuron 1 (x): [-1.09160765  0.99420337  3.95491422  0.2935302


📊 RMSE by Filter:
   EKF: 0.704986  |  UKF: 0.588753  |  PF: 0.136737

💡 To reproduce these results:
   Main: RANDOM_SEED = 59034
   DE Optim: DE_SEED = 71379

--- Optimized Parameter Summary ---
EKF params: Q=2.000e-04 R=8.000e-03 P0~1.078e+00 eta=0.400
UKF params: alpha=1.000e-02 eta=0.600 Qdiag=2.000e-04 R=8.000e-03
PF params: Q_std=[0.1, 0.1, 0.5] R_std=[0.1, 0.1, 0.5] ESS_th=400.0 n_particles=800

Optimization + simulation complete.


In [30]:
# Reconstruct reference trajectory for plotting
x_ref_hist = np.zeros_like(x_true)
for k, t in enumerate(t_history):
    x_ref_k, _, _ = reference_trajectory(t, ref_kind)
    x_ref_hist[k] = x_ref_k

fig_trajectory.add_trace(go.Scatter(x=x_ref_hist[:,0], y=x_ref_hist[:,1],
    mode='lines', name='Ref', line=dict(color='gray', width=2, dash='longdash')))
fig_trajectory.update_layout(title='Trajectory Tracking (Neural PID + RHONN feedback)')